# Model Training

This notebook builds a reproducible training pipeline for a Streamlit-based music recommender system. It loads the source dataset, preprocesses text data, creates bag-of-words features, computes cosine similarity, and saves the required artifacts.

**Generated artifacts**
- `df.pkl`: cleaned dataset used by the app
- `similarity.pkl`: cosine similarity matrix used for recommendations

## Imports

Import the required libraries for data handling, feature extraction, similarity computation, and artifact serialization.

In [ ]:
import pickle
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Data Loading

Load the source dataset from the project root.

In [ ]:
data_path = Path("spotify_millsongdata.csv")

if not data_path.exists():
    raise FileNotFoundError("Dataset not found. Please provide spotify_millsongdata.csv")

df = pd.read_csv(data_path)

## Data Preprocessing

Keep only the required columns (`song`, `artist`, `text`), remove missing values, and normalize text to lowercase for consistent feature extraction.

In [ ]:
df = df[["song", "artist", "text"]].dropna().copy()
df = df.drop_duplicates(subset=["song", "artist"])
df["text"] = df["text"].str.lower()

## Feature Engineering

Convert song lyrics into numerical features using `CountVectorizer` with a constrained vocabulary for efficiency.

In [ ]:
vectorizer = CountVectorizer(max_features=5000, stop_words="english")
text_vectors = vectorizer.fit_transform(df["text"])

## Similarity Computation

Compute pairwise cosine similarity across all songs using the generated text vectors.

In [ ]:
similarity = cosine_similarity(text_vectors)

## Validation Checks

Run final data and shape checks before writing artifacts to ensure compatibility with the Streamlit app.

In [ ]:
# Validation checks before saving artifacts

required_columns = {"song", "artist", "text"}
if not required_columns.issubset(df.columns):
    raise ValueError(f"Missing required columns: {required_columns - set(df.columns)}")

if len(df) == 0:
    raise ValueError("Dataframe is empty after preprocessing")

if similarity.shape[0] != len(df) or similarity.shape[1] != len(df):
    raise ValueError("Similarity matrix shape mismatch with dataframe")

print(f"Dataset size: {len(df)}")
print(f"Similarity shape: {similarity.shape}")

similarity = similarity.astype("float32")

print("Validation checks passed ✅")

## Saving Artifacts

Persist the cleaned dataframe and similarity matrix as pickle files so the Streamlit app can load them directly during inference.

In [ ]:
df = df[["song", "artist", "text"]]

with open("df.pkl", "wb") as df_file:
    pickle.dump(df, df_file)

with open("similarity.pkl", "wb") as similarity_file:
    pickle.dump(similarity, similarity_file)

print("Artifacts saved: df.pkl and similarity.pkl ✅")